In [1]:
import os, shutil, glob, subprocess, sys

zip_path = "/content/pettingzoo_project_conference_gpu_vectorized_v30_truth_social1_pgg25.zip"
project_dir = "/content/project"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pettingzoo", "gymnasium", "shimmy", "numpy", "pandas", "matplotlib", "torch"],
    check=True,
)

if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

os.makedirs(project_dir, exist_ok=True)
subprocess.run(["unzip", "-oq", zip_path, "-d", project_dir], check=True)

entries = [p for p in glob.glob(os.path.join(project_dir, "*")) if os.path.isdir(p)]
if len(entries) == 1 and os.path.exists(os.path.join(entries[0], "demo_run_conference.py")):
    os.chdir(entries[0])
else:
    os.chdir(project_dir)

import torch
print("Project dir:", os.getcwd(), flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), flush=True)

def run_live(cmd):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

print("\nRunning main conference experiment...\n", flush=True)
run_live([sys.executable, "-u", "demo_run_conference.py"])

print("\nInspecting outputs...\n", flush=True)
if os.path.exists("inspect_outputs.py"):
    run_live([sys.executable, "-u", "inspect_outputs.py"])
else:
    print("inspect_outputs.py not found, skipping.", flush=True)

print("\nComparing against saved synthetic truth artifacts...\n", flush=True)
if os.path.exists("compare_synthetic_ground_truth.py"):
    run_live([sys.executable, "-u", "compare_synthetic_ground_truth.py"])
else:
    print("compare_synthetic_ground_truth.py not found, skipping.", flush=True)

print("\nSaved non-ablation output files:\n", flush=True)
for path in sorted(
    p for p in glob.glob(os.path.join(os.getcwd(), "**", "*"), recursive=True)
    if os.path.isfile(p)
    and ("ablation" not in os.path.basename(p).lower())
    and (p.endswith(".csv") or p.endswith(".json") or p.endswith(".txt"))
):
    print(path)

Project dir: /content/project
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition

Running main conference experiment...

[conference] generating target moments
[conference] target env=rpd start
[evaluate_candidate] candidate=predictive_truth_homogeneous_social_tradeoff agent_kind=deep episodes=500 envs=['rpd']
  -> env=rpd starting
  <- env=rpd done rows=2400 moments=5
[conference] target env=rpd done total_target_moments=5
[conference] target env=ult start
[evaluate_candidate] candidate=predictive_truth_homogeneous_social_tradeoff agent_kind=deep episodes=500 envs=['ult']
  -> env=ult starting
/content/project/marl_utility_calibration/envs.py:436: RuntimeWarning: UltimatumEnv observation layout changed in ultimatum_repeated_v2 (shape 5 -> 4). Saved models or results from older versions are not directly compatible and should be rerun.
  return UltimatumEnv(rounds=rounds, **kwargs)
W0319 03:37:22.740000 2950 torch/_dynamo/convert_frame.py:1676] [0/8] torch._dynamo hi

In [2]:
import os, glob, zipfile
from google.colab import files

root = os.getcwd()
zip_name = "conference_non_ablation_results.zip"
zip_path = os.path.join(root, zip_name)

result_files = sorted(
    p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
    if os.path.isfile(p)
    and ("ablation" not in os.path.basename(p).lower())
    and (p.endswith(".csv") or p.endswith(".json") or p.endswith(".txt"))
)

if not result_files:
    raise FileNotFoundError("No non-ablation .csv/.json/.txt files found.")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in result_files:
        arcname = os.path.relpath(path, root)
        zf.write(path, arcname)

print("Created:", zip_path)
print("Included files:")
for p in result_files:
    print("-", os.path.relpath(p, root))

files.download(zip_path)

Created: /content/project/conference_non_ablation_results.zip
Included files:
- colab_one_cell.txt
- outputs_predictive_optimality/agreement_summary.csv
- outputs_predictive_optimality/approximate_predictive_optimality_definition.txt
- outputs_predictive_optimality/bootstrap_summary.csv
- outputs_predictive_optimality/conference_config.json
- outputs_predictive_optimality/cross_method_winners.csv
- outputs_predictive_optimality/exhaustive_restricted_ranking.csv
- outputs_predictive_optimality/finalist_pairwise_tests.csv
- outputs_predictive_optimality/finalist_repeated_test.csv
- outputs_predictive_optimality/finalist_summary.csv
- outputs_predictive_optimality/finalist_winner_frequency.csv
- outputs_predictive_optimality/method_consensus_summary.csv
- outputs_predictive_optimality/optimality_certificates.csv
- outputs_predictive_optimality/overall_ranking.csv
- outputs_predictive_optimality/restart_variance_summary.csv
- outputs_predictive_optimality/restart_winner_frequency.csv
- out

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>